In [1]:
print("hello world")

hello world


In [1]:
import pandas as pd
xlsx_path = "./Job_simplified_myself.xlsx"
df = pd.read_excel(xlsx_path)
print(df.shape)
print(df.columns)
df.head()

(214, 84)
Index(['serial_no', 'Media site', 'Company name', 'Company logo', 'Title',
       'Catchphrase\n', 'Salary type', 'Salary', 'Salary details\n',
       'Employment Type\n', 'Job Industry', 'Job Category',
       'Social insurances', 'Job Benefits Details1', 'Job Benefits Details2',
       'Holidays & Leaves Details', 'Description', 'Requirements',
       'Requirements summary', 'Service Form', 'Working hours\n',
       'One day work details', 'Nearest Station', 'Nearest station access',
       'Selection flow\n', 'Recruiter message\n', 'Postal Code',
       'Address details', 'Google Map Url', 'Trial period duration\n',
       'Trial period details\n', 'Trial period salary',
       'Trial period working hours\n', 'Tag_出社勤務', 'Tag_在宅勤務 ', 'Tag_リモート相談可',
       'Tag_フレックスタイム制度', 'Tag_直行直帰', 'Tag_時短勤務', 'Tag_週4日勤務', 'Tag_反響営業のみ',
       'Tag_ノルマなし', 'Tag_追加の部分', 'Tag_インセンティブあり', 'Tag_高収入', 'Tag_高歩合率',
       'Tag_資格手当あり', 'Tag_昇給年1回以上', 'Tag_追加の部分.1', 'Tag_完全週休2日制', 'Tag_土日祝休み',


,serial_no,Media site,Company name,Company logo,Title,Catchphrase\n,Salary type,Salary,Salary details\n,Employment Type\n,...,Tag_管理職候補,Tag_独立支援制度あり,Tag_新規事業立ち上げメンバー,Tag_キャリアアップ可能,Tag_追加の部分.6,Tag_ネイルOK,Tag_髪色自由,Tag_服装自由,Tag_追加の部分.7,image data
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,Fudosanworks,株式会社バンダイ,NaN,不動産事務／地域密着・ネイル自由✨にぎやかオフィスの“秘密兵器”募集！,髪色・ネイルOKで“自分らしさ”も働きやすさも両方ゲット♪PC作業から外での撮影まで、動きの...,月給,230000,◇固定残業代について\n◎月給には固定残業代2万円分（月13時間分）を含みます。\n◎固定残...,正社員,...,False,False,False,False,False,False,True,True,False,NaN
4,NaN,Fudosanworks,株式会社バンダイ,NaN,〖不動産営業〗定時退社＆オフ充実！ブラック神話を覆す稼げる営業,年収800万も、夢じゃない⁉インセンティブで叶える未来、つくってます。（マジで。）\n,月給,280000,"※頑張りしだいで年収1,000万円以上も可能\n\n【インセンティブ】\n月ノルマ85万円を...",正社員,...,False,False,False,False,False,False,False,False,False,NaN


# Json dataset creation from xlsx file

In [11]:
import pandas as pd
import json

# ---------- CONFIG ----------
xlsx_path = "./Copy of Job_simplified_myself_Eitate thik thak ase sob_without_check_box.xlsx"
max_row_number = 188
output_path = "./jobs_output.json"
SERIAL_COL = "serial_no"   # ✅ correct column name
# ----------------------------

# ---------- READ & CLEAN ----------
df = pd.read_excel(xlsx_path)

df.columns = df.columns.str.strip()
df = df.dropna(how="all").reset_index(drop=True)

max_row_number = min(max_row_number, len(df) - 1)

# ---------- HELPER ----------
def is_true(value):
    if pd.isna(value):
        return False
    if isinstance(value, bool):
        return value
    if isinstance(value, (int, float)):
        return value == 1
    if isinstance(value, str):
        return value.strip().upper() in ["TRUE", "YES", "Y", "1"]
    return False

# ---------- BUILD JSON ----------
results = []
tag_columns = [c for c in df.columns if c.startswith("Tag_")]

for idx in range(0, max_row_number + 1):
    row = df.iloc[idx]
    row_json = {}
    tag_values = []

    for col in df.columns:
        value = row[col]

        # 🔥 FORCE serial_no → int here
        if col == SERIAL_COL:
            row_json[col] = None if pd.isna(value) else int(value)

        elif col in tag_columns:
            if is_true(value):
                tag_values.append(col.replace("Tag_", "").strip())

        else:
            row_json[col] = None if pd.isna(value) else value

    row_json["Tag"] = tag_values
    results.append(row_json)

# ---------- WRITE JSON ----------
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print(f"✅ JSON file successfully created: {output_path}")


✅ JSON file successfully created: ./jobs_output.json


# Salary splitting

In [2]:
import pandas as pd
# Read JSON file
df = pd.read_json("./jobs_output.json")
# Display the first few rows
print(df.head())

   serial_no    Media site  Company name  Company logo  \
0          2  Fudosanworks      株式会社バンダイ           NaN   
1          3  Fudosanworks      株式会社バンダイ           NaN   
2          4  Fudosanworks   株式会社ケイアイホーム           NaN   
3          5  Fudosanworks  株式会社アーバン企画開発           NaN   
4          6  Fudosanworks   株式会社サンエイホーム           NaN   

                                 Title  \
0  不動産事務／地域密着・ネイル自由✨にぎやかオフィスの“秘密兵器”募集！   
1      〖不動産営業〗定時退社＆オフ充実！ブラック神話を覆す稼げる営業   
2  ルームアドバイザー　未経験オーケーの不動産営業。9割反響で月30万も◎   
3       不動産賃貸のルームアドバイザー　雑談力が最強スキルになる仕事   
4      ルームアドバイザー　全集中！接客の呼吸　壱ノ型　完全分業ッ！！   

                                         Catchphrase Salary type  \
0  髪色・ネイルOKで“自分らしさ”も働きやすさも両方ゲット♪PC作業から外での撮影まで、動きの...          月給   
1        年収800万も、夢じゃない⁉インセンティブで叶える未来、つくってます。（マジで。）\n          月給   
2            入社1年で店長昇格。10年目で月収300万も。努力が“返ってくる”不動産営業。          月給   
3         気づけば通帳が笑ってる⁉転職ストーリー、一発逆転エンド！信じたのは、この環境でした。          月給   
4  20代でも年収500万円超え?!宅建20万円の祝い金＋定時退社＋毎年ケーキ支給されちゃう不動...      

In [3]:
print(df.columns)

Index(['serial_no', 'Media site', 'Company name', 'Company logo', 'Title',
       'Catchphrase', 'Salary type', 'Salary', 'Salary details',
       'Employment Type', 'Job Industry', 'Job Category', 'Social insurances',
       'Job Benefits Details1', 'Job Benefits Details2',
       'Holidays & Leaves Details', 'Description', 'Requirements',
       'Requirements summary', 'Service Form', 'Working hours',
       'One day work details', 'Nearest Station', 'Nearest station access',
       'Selection flow', 'Recruiter message', 'Postal Code', 'Address details',
       'Google Map Url', 'Trial period duration', 'Trial period details',
       'Trial period salary', 'Trial period working hours', 'image data',
       'Tag'],
      dtype='object')


In [5]:
print(len(df))
print(df.shape)

184
(184, 35)


In [6]:
first_row = df.iloc[0]

In [11]:
print(first_row['Salary'])

230000


In [12]:
print( df['Salary'].unique())

[230000 280000 240000 '300000～450000' '230000～360000' 1020 '300000～'
 '250000～400000' 270000 '233290～281580' '350000～420000' '235000～450000'
 '280000～400000' 250000 265000 '270000～400000' '240000～400000'
 '420000～570000' '265000～500000' '280000～' '250000～350000' '200000～'
 '305,000～460,000円' '270,000～450,000円' '280,000～450,000円'
 '250,000～350,000円' '300,000～800,000円' '285,000円' '278,000～302,000円'
 '290,000～350,000円' '260,000～360,000円' '300,000円～1,500,000円'
 '260,500～290,000円' '4,000,000～5,500,000円' '300,000～660,000円'
 '360,000～430,000円' '350,000～500,000円' '290,000円～' '300,000円'
 '380,000～500,000円' '309,550円～' '300,000～500,000円' '250,000～466,000円'
 '330,000円' '280,000～800,000円' '270,000～800,000円' '250,000～800,000円'
 '272,000円' '285,000～350,000円' '265,000～300,000円' '280,000円'
 '329,000～436,000円' '250,000～300,000円' '270,000～336,000円'
 '300,000～390,000円' '260,000～410,000円' '285,715～580,000円'
 '300,000～450,000円' '240,000～250,000円' '250,000円～' '266,000～540,000円'
 '300,000円～' '250,000～320,000

In [16]:
def salary_splitting(salary):
    """
    Splits a salary string into min_salary and max_salary.
    - Single values go into min_salary, max_salary = None
    - Ranges are split on '～'
    - Handles commas and '円'
    """
    if salary is None:
        return (None, None)

    # Convert to string and clean
    salary = str(salary).replace(" ", "").replace("円", "").replace(",", "")

    # Check for range
    if '～' in salary:
        parts = salary.split('～')
        try:
            min_salary = int(parts[0]) if parts[0] else None
        except:
            min_salary = None
        try:
            max_salary = int(parts[1]) if parts[1] else None
        except:
            max_salary = None
        return (min_salary, max_salary)

    # Single number → only min_salary
    try:
        min_salary = int(salary)
        return (min_salary, None)
    except:
        return (None, None)



max_salary,min_salary=salary_splitting("305,000～460,000円")
print(f"max_salary===={max_salary} min_salary====={min_salary}")
max_salary,min_salary=salary_splitting("250000")
print(f"max_salary===={max_salary} min_salary====={min_salary}")

max_salary,min_salary=salary_splitting("300000～")
print(f"max_salary===={max_salary} min_salary====={min_salary}")

max_salary,min_salary=salary_splitting("300000～450000")
print(f"max_salary===={max_salary} min_salary====={min_salary}")




max_salary====305000 min_salary=====460000
max_salary====250000 min_salary=====None
max_salary====300000 min_salary=====None
max_salary====300000 min_salary=====450000


In [2]:
def salary_splitting(salary):
    """
    Splits a salary string into min_salary and max_salary.
    - Single values go into min_salary, max_salary = None
    - Ranges are split on '～'
    - Handles commas and '円'
    """
    if salary is None:
        return (None, None)

    # Convert to string and clean
    salary = str(salary).replace(" ", "").replace("円", "").replace(",", "")

    # Check for range
    if '～' in salary:
        parts = salary.split('～')

        # If both values exist → proper range
        if parts[0] and len(parts) > 1 and parts[1]:
            try:
                min_salary = int(parts[0])
            except:
                min_salary = None
            try:
                max_salary = int(parts[1])
            except:
                max_salary = None
            return (min_salary, max_salary)

        # Otherwise → treat as single value (min only)
        try:
            min_salary = int(parts[0] or parts[1])
            return (min_salary, None)
        except:
            return (None, None)

    # Single number → only min_salary
    try:
        min_salary = int(salary)
        return (min_salary, None)
    except:
        return (None, None)

        
min_salary, max_salary = salary_splitting("305,000～460,000円")
print(f"min_salary===={min_salary} max_salary====={max_salary}")

min_salary, max_salary = salary_splitting("250000")
print(f"min_salary===={min_salary} max_salary====={max_salary}")

min_salary, max_salary = salary_splitting("300000～")
print(f"min_salary===={min_salary} max_salary====={max_salary}")

min_salary, max_salary = salary_splitting("300000～450000")
print(f"min_salary===={min_salary} max_salary====={max_salary}")


min_salary====305000 max_salary=====460000
min_salary====250000 max_salary=====None
min_salary====300000 max_salary=====None
min_salary====300000 max_salary=====450000
